In [ ]:
# --- Creativity @ answer-level: subtask × model panels, colored by generation, markers by seed ---
import os, re, math
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from matplotlib.lines import Line2D
from matplotlib.colors import Normalize
from matplotlib.collections import LineCollection

# ======= 配置：两张 index CSV（里面有 csv_file_path 列）======
INDEX_FILES = {
    "oscillator1": "/Users/lievretre/Desktop/evo_eval_sheets/oscillator1_raw.csv",
    "oscillator2": "/Users/lievretre/Desktop/evo_eval_sheets/oscillator2_raw.csv",
}

# 只画 child
CHILD_ONLY = False

# 列名候选
PATH_COL_CANDS  = ["csv_file_path", "csv_path", "path"]
GEN_COL_CANDS   = ["generation", "gen", "g"]
TYPE_COL_CANDS  = ["type", "sample_type", "role"]
FITN_COL_CANDS  = ["fitness_normed"]

# ✅ 改：优先找“原始多样性/距离”列，其次才用已经归一化的列兜底
DIVERSITY_RAW_CANDS = [
    "total_distance", "novelty", "diversity", "distance"
]
DIVERSITY_NORMED_FALLBACK = [
    "total_distance_normed", "novelty_normed"
]

MODEL_COL_CANDS = ["model", "alias", "model_alias", "price_key"]
SEED_COL_CANDS  = ["seed", "seed_id", "run"]

# 视觉：白底、紧凑
mpl.rcParams.update(mpl.rcParamsDefault)
plt.style.use("default")
mpl.rcParams["figure.facecolor"]  = "white"
mpl.rcParams["axes.facecolor"]    = "white"
mpl.rcParams["savefig.facecolor"] = "white"

# seed → marker
SEED_MARKERS = ['o', 's', '^', 'D', 'P', 'X', 'v', '<', '>', '*', 'h', 'H', 'p']
def seed_to_marker(seed_value) -> str:
    try:
        i = int(seed_value)
    except Exception:
        return 'o'
    return SEED_MARKERS[i % len(SEED_MARKERS)]

def find_col(df: pd.DataFrame, cands: List[str], required: bool=True) -> Optional[str]:
    for c in cands:
        if c in df.columns:
            return c
    if required:
        raise ValueError(f"Required column not found. Tried: {cands}")
    return None

def find_first_column(df: pd.DataFrame, cands: List[str]) -> Optional[str]:
    for c in cands:
        if c in df.columns:
            return c
    return None

def try_get_seed_series(df: pd.DataFrame, path: str) -> pd.Series:
    for c in SEED_COL_CANDS:
        if c in df.columns:
            return df[c]
    name = Path(path).name
    m = re.search(r"seed[_\-]?(\d+)", name, flags=re.IGNORECASE)
    if m:
        v = m.group(1)
        return pd.Series([int(v)]*len(df), index=df.index)
    return pd.Series(["unknown"]*len(df), index=df.index)

def load_index(index_path: str) -> pd.DataFrame:
    p = Path(index_path)
    idx = pd.read_csv(p)
    path_col = find_col(idx, PATH_COL_CANDS, required=True)
    keep = [path_col]
    for c in MODEL_COL_CANDS + SEED_COL_CANDS:
        if c in idx.columns:
            keep.append(c)
    out = idx[keep].copy()
    out.rename(columns={path_col: "csv_file_path"}, inplace=True)
    return out

def read_process_csv(process_csv_path: str) -> Optional[pd.DataFrame]:
    p = Path(process_csv_path)
    if not p.exists():
        print(f"[warn] missing process CSV: {p}")
        return None
    try:
        return pd.read_csv(p)
    except Exception as e:
        print(f"[warn] read failed: {p} -> {e}")
        return None

def pareto_front_xy(x: np.ndarray, y: np.ndarray, direction: str = "ltr") -> np.ndarray:
    ok = np.isfinite(x) & np.isfinite(y)
    if not ok.any():
        return np.empty((0, 2))
    xv = x[ok]; yv = y[ok]
    if direction == "rtl":
        order = np.argsort(xv)[::-1]          # x 降序
        xv = xv[order]; yv = yv[order]
        best = -np.inf
        pts = []
        for xi, yi in zip(xv, yv):
            if yi > best:
                pts.append((xi, yi)); best = yi
        return np.array(pts)
    else:
        order = np.argsort(xv)                 # x 升序
        xv = xv[order]; yv = yv[order]
        best = -np.inf
        pts = []
        for xi, yi in zip(xv, yv):
            if yi > best:
                pts.append((xi, yi)); best = yi
        return np.array(pts)

# ======= 第一步：读取并收集“原始多样性值” =======
panels: Dict[Tuple[str,str], pd.DataFrame] = {}
all_div_raw = []      # 收集全局的 raw diversity 值
used_fallback = False # 是否使用了已归一化列作为原始兜底

for subtask, index_csv in INDEX_FILES.items():
    idx = load_index(index_csv)
    for _, row in idx.iterrows():
        path = str(row["csv_file_path"]).strip()
        df = read_process_csv(path)
        if df is None or df.empty:
            continue

        gen_col  = find_col(df, GEN_COL_CANDS, required=True)
        fit_col  = find_col(df, FITN_COL_CANDS, required=True)

        # 优先找原始列；找不到就用已归一化列兜底
        div_raw_col = find_first_column(df, DIVERSITY_RAW_CANDS)
        if div_raw_col is None:
            div_raw_col = find_first_column(df, DIVERSITY_NORMED_FALLBACK)
            if div_raw_col is None:
                print(f"[warn] {path} has no diversity/novelty column (raw or normed).")
                continue
            used_fallback = True

        type_col = find_col(df, TYPE_COL_CANDS, required=False)
        model_col= find_col(df, MODEL_COL_CANDS, required=False)

        # 仅 child
        if CHILD_ONLY and type_col is not None:
            df = df[df[type_col].astype(str).str.lower().eq("child")].copy()
        if df.empty:
            continue

        seed_series = try_get_seed_series(df, path)

        use_cols = [gen_col, fit_col, div_raw_col]
        if model_col: use_cols.append(model_col)
        sub = df[use_cols].copy()
        sub.rename(columns={
            gen_col      : "generation",
            fit_col      : "fitness_normed",
            div_raw_col  : "diversity_raw",   # 👈 统一命名为“原始多样性值”
        }, inplace=True)
        sub["seed"] = seed_series.values
        sub["subtask"] = subtask

        # 模型名（尽力拿）
        if model_col and model_col in df.columns and df[model_col].notna().any():
            model_name = str(df[model_col].dropna().iloc[0])
        else:
            model_name = None
            for c in MODEL_COL_CANDS:
                if c in row and isinstance(row[c], str) and row[c].strip():
                    model_name = row[c]; break
            if not model_name:
                model_name = Path(path).stem

        key = (subtask, model_name)
        panels.setdefault(key, pd.DataFrame())
        panels[key] = pd.concat([panels[key], sub], ignore_index=True)

        # 收集全局 raw
        all_div_raw.append(pd.to_numeric(sub["diversity_raw"], errors="coerce"))

# ======= 第二步：按 subtask 归一（也保留全局可选） =======
NORM_SCOPE = "by_subtask"   # 可选: "global" / "by_subtask" / "by_panel"
ROBUST_Q = (1, 99)            # 例如 (1, 99) 走分位数鲁棒归一；None 则用 min–max
EXCLUDE_0_1 = True           # 新增参数：是否排除 diversity_raw == 0 或 1 的极端值

# 先做清洗（与原来一致）
cleaned_panels = {}
for k, df in list(panels.items()):
    if df.empty:
        continue
    df["generation"]     = pd.to_numeric(df["generation"], errors="coerce")
    df["fitness_normed"] = pd.to_numeric(df["fitness_normed"], errors="coerce")
    df["diversity_raw"]  = pd.to_numeric(df["diversity_raw"],  errors="coerce")
    df = df.dropna(subset=["generation", "fitness_normed", "diversity_raw"])
    df = df.sort_values("generation").reset_index(drop=True)
    cleaned_panels[k] = df
panels = cleaned_panels

if not panels:
    print("[warn] no diversity values found; abort plotting.")
else:

    # ========= 定义一个统一的小函数用于取上下界 =========
    def get_lo_hi(vec):
        vec = vec.astype(float)
        if EXCLUDE_0_1:
            vec = vec[(vec != 0.0) & (vec != 1.0)]
        if len(vec) == 0:
            return 0.0, 1.0
        if ROBUST_Q:
            lo, hi = np.nanpercentile(vec, ROBUST_Q)
        else:
            lo, hi = np.nanmin(vec), np.nanmax(vec)
        if not (np.isfinite(lo) and np.isfinite(hi)) or hi <= lo:
            lo, hi = 0.0, 1.0
        return lo, hi

    # ========= 三种归一化模式 =========
    if NORM_SCOPE == "global":
        vec = pd.concat([df["diversity_raw"] for df in panels.values()],
                        ignore_index=True).astype(float)
        lo, hi = get_lo_hi(vec)
        den = hi - lo
        for k, df in panels.items():
            df["diversity_gnormed"] = ((df["diversity_raw"] - lo) / den).clip(0.0, 1.0)

    elif NORM_SCOPE == "by_subtask":
        # k = (subtask, model_name)
        per_subtask_vals = defaultdict(list)
        for (subtask, _m), df in panels.items():
            per_subtask_vals[subtask].append(df["diversity_raw"])

        subtask_minmax = {}
        for subtask, series_list in per_subtask_vals.items():
            vec = pd.concat(series_list, ignore_index=True).astype(float)
            lo, hi = get_lo_hi(vec)
            subtask_minmax[subtask] = (lo, hi)

        for (subtask, _m), df in panels.items():
            lo, hi = subtask_minmax[subtask]
            den = hi - lo
            df["diversity_gnormed"] = ((df["diversity_raw"] - lo) / den).clip(0.0, 1.0)

    elif NORM_SCOPE == "by_panel":
        for k, df in panels.items():
            vec = df["diversity_raw"].astype(float)
            lo, hi = get_lo_hi(vec)
            den = hi - lo
            df["diversity_gnormed"] = ((df["diversity_raw"] - lo) / den).clip(0.0, 1.0)

    else:
        raise ValueError(f"Unknown NORM_SCOPE: {NORM_SCOPE}")

# ======= 第三步：画所有 (subtask, model) 子图 =======
keys = sorted(panels.keys(), key=lambda x: (x[0], str(x[1]).lower()))
n = len(keys)
if n == 0:
    print("[warn] no data to plot.")
else:
    ncols = min(5, n)
    nrows = int(math.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.0*ncols, 2.6*nrows), squeeze=False)
    fig.suptitle("Symbolic Regression (Oscillator1 / Oscillator2) — Novelty at the Answer Level",
                 fontsize=16, y=0.995)

    # generation 全局范围用于 colorbar
    g_all = pd.concat([panels[k]["generation"] for k in keys], ignore_index=True)
    gmin_all, gmax_all = g_all.min(), g_all.max()
    cmap = plt.get_cmap("viridis")
    gnorm_global = Normalize(vmin=gmin_all, vmax=gmax_all)

    for i, key in enumerate(keys):
        r, c = divmod(i, ncols)
        ax = axes[r][c]
        df = panels[key]
        subtask, model = key

        # 颜色：按全局 generation 范围映射
        colors = cmap(gnorm_global(df["generation"].to_numpy()))

        # seed 分 marker
        seeds = df["seed"].astype(str).fillna("unknown")
        uniq_seeds = seeds.unique().tolist()

        for s in uniq_seeds:
            mask = seeds.eq(s).to_numpy()
            mk = seed_to_marker(s)
            ax.scatter(df.loc[mask, "fitness_normed"], df.loc[mask, "diversity_gnormed"],
                       s=18, marker=mk, c=colors[mask], edgecolor="white", linewidth=0.35, alpha=0.95, zorder=2)

        # # —— 中位数进化轨迹（跨 seed 聚合；使用全局归一后的 y）——
        # g_med = (df.groupby('generation', as_index=False)
        #            .agg(fit_med=('fitness_normed', 'median'),
        #                 nov_med=('diversity_gnormed', 'median'))
        #            .sort_values('generation'))
        # if len(g_med) >= 2:
        #     ax.plot(g_med['fit_med'], g_med['nov_med'],
        #             color='lightsalmon', lw=1.6, alpha=0.85, zorder=3, label='_nolegend_')
        #     ax.scatter(g_med.iloc[[0]]['fit_med'],  g_med.iloc[[0]]['nov_med'],
        #                s=24, marker='o', facecolor='none', edgecolor='black', lw=0.7, zorder=3)
        #     ax.scatter(g_med.iloc[[-1]]['fit_med'], g_med.iloc[[-1]]['nov_med'],
        #                s=28, marker='*', facecolor='black', edgecolor='white', lw=0.5, zorder=3)

        # —— Pareto frontier（x=fitness_normed, y=diversity_gnormed；从右往左）——
        pts = pareto_front_xy(
            df["fitness_normed"].to_numpy(float),
            df["diversity_gnormed"].to_numpy(float),
            direction="rtl"
        )
        if pts.size:
            ax.plot(pts[:,0], pts[:,1], '-', color="steelblue", lw=1.5, alpha=0.85, zorder=1.5)

        # 轴：固定 0~1，白底，淡网格
        ax.set_xlim(-0.02, 1.02)
        ax.set_ylim(-0.02, 1.02)
        ax.grid(True, color="#eee", linewidth=0.8, zorder=0)

        # 只在最左列/最下行显示坐标轴标签
        if c == 0:
            ax.set_ylabel("Normalized diversity")
        else:
            ax.set_yticklabels([])
        if r == nrows - 1:
            ax.set_xlabel("Normalized fitness")
        else:
            ax.set_xticklabels([])

        # 标题
        ax.set_title(f"{subtask} | {model}", fontsize=9.5, pad=3)

    # 关闭多余子图
    for j in range(n, nrows*ncols):
        r, c = divmod(j, ncols)
        axes[r][c].set_axis_off()

    # 全局 generation colorbar（右侧）
    cax = fig.add_axes([0.92, 0.20, 0.015, 0.60])
    cb = mpl.colorbar.ColorbarBase(cax, cmap=cmap, norm=gnorm_global)
    cb.set_label("Generation", rotation=90)

    fig.subplots_adjust(left=0.06, right=0.90, top=0.94, bottom=0.08, wspace=0.16, hspace=0.20)
    plt.savefig("/Users/lievretre/Desktop/evo_eval_plots_10_28/novelty_scatter_sr.png",dpi=300)
    plt.show()

In [ ]:
# === Symbolic Regression — Task-level MDS (distance = 1 - cos) ===
# 输入：多个 subtask 的 index.csv（每个 index 里列出要遍历的 CSV）
# 要求：每个 process CSV 里至少有：generation / fitness(or fitness_normed) / embedding(JSON)
# 输出：每个 (subtask, model) 一个面板，颜色=generation，大小=fitness_normed

import os, re, json, math
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from sklearn.manifold import MDS

# -------------------------
# 你的 index 文件字典（按子任务）
# -------------------------
INDEX_FILES = {
    "oscillator1": "/Users/lievretre/Desktop/evo_eval_sheets/oscillator1_raw.csv",
    "oscillator2": "/Users/lievretre/Desktop/evo_eval_sheets/oscillator2_raw.csv",
}

# 只画 child（可选）
CHILD_ONLY = False

# 列名候选（自动探测）
PATH_COL_CANDS   = ["csv_file_path", "csv_path", "path"]
GEN_COL_CANDS    = ["generation", "gen", "g"]
TYPE_COL_CANDS   = ["type", "sample_type", "role"]
FITN_COL_CANDS   = ["fitness_normed", "fitness", "score", "objective", "loss"]
MODEL_COL_CANDS  = ["model", "alias", "model_alias", "price_key", "engine"]
EMB_COL_CANDS    = ["embedding", "behavior", "vector", "embed"]

# MDS 超参（与 TSP 对齐）
MDS_MAX_POINTS   = 4000      # 超过则抽样学平面 + OOS 插值
PER_BUCKET       = 60        # (model, generation) 每桶上限（用于抽样）
MDS_KW = dict(
    n_components=2,
    dissimilarity="precomputed",
    n_init=1,
    max_iter=300,
    eps=1e-3,
    random_state=42,
    verbose=1,
)

# OOS 插值：KNN & 权重
OOS_K = 8
OOS_P = 2.0     # 权重=1/(d+1e-8)^P
RNG = np.random.default_rng(42)

# 预计算距离矩阵缓存（可选）
USE_PRECOMPUTED = True
DIST_CACHE_DIR  = ".sr_mds_dist_cache"     # 保存每个 subtask 的 D_fit（基点距离矩阵）
Path(DIST_CACHE_DIR).mkdir(parents=True, exist_ok=True)

# 视觉
mpl.rcParams.update(mpl.rcParamsDefault)
plt.style.use("default")
mpl.rcParams["figure.facecolor"]  = "white"
mpl.rcParams["axes.facecolor"]    = "white"
mpl.rcParams["savefig.facecolor"] = "white"

# ========== 通用工具 ==========
def find_col(df: pd.DataFrame, cands: List[str], required: bool=True) -> Optional[str]:
    for c in cands:
        if c in df.columns:
            return c
    if required:
        raise ValueError(f"Required column not found. Tried: {cands}")
    return None

def load_index(index_path: str) -> pd.DataFrame:
    idx = pd.read_csv(index_path)
    path_col = find_col(idx, PATH_COL_CANDS, required=True)
    keep = [path_col]
    for c in MODEL_COL_CANDS:
        if c in idx.columns: keep.append(c)
    out = idx[keep].copy()
    out.rename(columns={path_col: "csv_file_path"}, inplace=True)
    return out

def read_csv_maybe(path: str) -> Optional[pd.DataFrame]:
    p = Path(path)
    if not p.exists():
        print(f"[warn] missing CSV: {p}")
        return None
    try:
        return pd.read_csv(p)
    except Exception as e:
        print(f"[warn] read failed: {p} -> {e}")
        return None

def parse_embedding(x):
    """把 embedding 列解析为 np.array[float32]，失败返回 None"""
    if isinstance(x, (list, np.ndarray)):
        arr = np.asarray(x, dtype=np.float32)
        return arr if arr.ndim==1 and arr.size>0 else None
    if isinstance(x, str):
        try:
            v = json.loads(x)
            arr = np.asarray(v, dtype=np.float32)
            return arr if arr.ndim==1 and arr.size>0 else None
        except Exception:
            return None
    return None

def robust_minmax(x: np.ndarray, q=(1,99)):
    x = np.asarray(x, dtype=float)
    if len(x)==0:
        return x
    lo, hi = np.nanpercentile(x, q)
    den = (hi-lo) if hi>lo else 1.0
    y = (x - lo) / den
    return np.clip(y, 0, 1)

def stratified_sample_idx(df: pd.DataFrame, per_bucket=PER_BUCKET, total_cap=MDS_MAX_POINTS) -> np.ndarray:
    parts = []
    for (m, g), sub in df.groupby(["model","generation"], sort=False):
        if len(sub) > per_bucket:
            parts.append(sub.sample(per_bucket, random_state=int(RNG.integers(1<<31))))
        else:
            parts.append(sub)
    out = pd.concat(parts, ignore_index=False)
    if len(out) > total_cap:
        out = out.sample(total_cap, random_state=42)
    return out.index.to_numpy()

# ======= 余弦距离（1 - cos），加速型实现 =======
def l2_normalize_rows(A: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    n = np.linalg.norm(A, axis=1, keepdims=True)
    n = np.where(n>eps, n, 1.0)
    return A / n

def cosine_D_fit(A_fit: np.ndarray) -> np.ndarray:
    """
    A_fit: (m, P) float32，已按行 L2 归一化
    返回 D_fit: (m, m) float32，D = 1 - (A_fit @ A_fit^T)（不取绝对值）
    """
    S = np.clip(A_fit @ A_fit.T, -1.0, 1.0).astype(np.float32)
    D = (1.0 - S).astype(np.float32)
    np.fill_diagonal(D, 0.0)
    return D

def cosine_D_query_to_fit(Aq: np.ndarray, Af: np.ndarray, block: int = 4000) -> np.ndarray:
    """
    Aq: (b, P)  Af: (m, P)  都需已 L2 归一化
    返回 Dq: (b, m) float32
    分块避免显存/内存峰值
    """
    b = Aq.shape[0]; m = Af.shape[0]
    out = np.zeros((b, m), dtype=np.float32)
    for s in range(0, b, block):
        sl = slice(s, min(b, s+block))
        S = np.clip(Aq[sl] @ Af.T, -1.0, 1.0).astype(np.float32)
        out[sl] = 1.0 - S
    return out

def oos_place_by_knn_cos(A_all, fit_idx, Y_fit, k=8, p=2.0, block=4000):
    """
    A_all: (N,P) 全体向量（未归一化/已归一化均可，本函数会归一化）
    fit_idx: (m,) 基点索引
    Y_fit: (m,2) MDS 结果
    返回 Y: (N,2)
    """
    N = A_all.shape[0]
    Y = np.zeros((N, 2), dtype=np.float32)
    Y[fit_idx] = Y_fit

    mask = np.ones(N, dtype=bool)
    mask[fit_idx] = False
    q_idx = np.where(mask)[0]
    if len(q_idx) == 0:
        return Y

    Af = l2_normalize_rows(A_all[fit_idx].astype(np.float32, copy=False))
    for s in range(0, len(q_idx), block):
        ids = q_idx[s: s+block]
        Aq  = l2_normalize_rows(A_all[ids].astype(np.float32, copy=False))
        Dq  = cosine_D_query_to_fit(Aq, Af, block=8000)        # (b, m)

        # 取每行的 k 近邻
        m = Af.shape[0]
        if k < m:
            nn = np.argpartition(Dq, kth=k, axis=1)[:, :k]
            rows = np.arange(len(ids))[:, None]
            dnn = Dq[rows, nn]
        else:
            nn = np.tile(np.arange(m), (len(ids),1))
            dnn = Dq

        w = 1.0 / (dnn + 1e-8) ** p
        w = w / w.sum(axis=1, keepdims=True)
        Y[ids] = (w[..., None] * Y_fit[nn]).sum(axis=1).astype(np.float32)

    return Y

# ========== 主流程：按 subtask 学一个平面，再分模型面板 ==========
all_panels: Dict[Tuple[str,str], pd.DataFrame] = {}

for subtask, index_csv in INDEX_FILES.items():
    print(f"\n=== Subtask: {subtask} ===")
    idx = load_index(index_csv)

    rows = []
    for _, row in idx.iterrows():
        csv_path = str(row["csv_file_path"]).strip()
        df = read_csv_maybe(csv_path)
        if df is None or df.empty: 
            continue

        # 找列
        gen_col   = find_col(df, GEN_COL_CANDS, required=True)
        fit_col   = find_col(df, FITN_COL_CANDS, required=True)
        emb_col   = find_col(df, EMB_COL_CANDS, required=True)
        type_col  = find_col(df, TYPE_COL_CANDS, required=False)
        model_col = find_col(df, MODEL_COL_CANDS, required=False)

        # 过滤 child
        if CHILD_ONLY and type_col and type_col in df.columns:
            df = df[df[type_col].astype(str).str.lower().eq("child")].copy()
            if df.empty: 
                continue

        # 拿模型名（优先来自 CSV 行）
        if model_col and model_col in df.columns and df[model_col].notna().any():
            model_name = str(df[model_col].dropna().iloc[0])
        else:
            # 退回 index 行里的字段
            model_name = None
            for c in MODEL_COL_CANDS:
                if c in row and isinstance(row[c], str) and row[c].strip():
                    model_name = row[c]; break
            if not model_name:
                model_name = Path(csv_path).stem

        # 取必要列
        sub = pd.DataFrame({
            "generation": pd.to_numeric(df[gen_col], errors="coerce"),
            "fitness_raw": pd.to_numeric(df[fit_col], errors="coerce"),
            "embedding_raw": df[emb_col],
            "model": model_name,
        })
        if type_col and type_col in df.columns:
            sub["type"] = df[type_col].astype(str)
        else:
            sub["type"] = "unknown"

        # 解析 embedding
        sub["emb"] = sub["embedding_raw"].apply(parse_embedding)
        sub = sub.dropna(subset=["generation", "fitness_raw", "emb"]).reset_index(drop=True)
        if sub.empty:
            continue

        rows.append(sub[["generation","fitness_raw","emb","model","type"]])

    if not rows:
        print(f"[warn] no rows collected for {subtask}")
        continue

    big = pd.concat(rows, ignore_index=True)
    big = big.reset_index(drop=True)
    print(f"[info] collected rows: {len(big)} | models: {big['model'].nunique()}")

    # 把可变长 emb 拼成矩阵（过滤掉长度不一致的）
    lens = big["emb"].apply(lambda a: len(a))
    mode_len = int(lens.mode().iloc[0])      # 取众数长度
    big = big[lens.eq(mode_len)].copy().reset_index(drop=True)
    if big.empty:
        print(f"[warn] embeddings empty after length filtering for {subtask}")
        continue

    A = np.stack(big["emb"].tolist()).astype(np.float32)    # (N, P)
    N, P = A.shape
    print(f"[info] embedding matrix: {A.shape}")

    # ---- 学任务级平面 ----
    # 抽样学 MDS（防爆），其余用 OOS 插值
    if len(big) <= MDS_MAX_POINTS:
        fit_idx = np.arange(len(big))
    else:
        fit_idx = stratified_sample_idx(big, per_bucket=PER_BUCKET, total_cap=MDS_MAX_POINTS)
    fit_mask = np.zeros(len(big), dtype=bool); fit_mask[fit_idx] = True

    Af = l2_normalize_rows(A[fit_idx])              # (m, P)
    cache_key = f"{subtask}_m{len(fit_idx)}_p{P}_cos.npy"
    cache_path = Path(DIST_CACHE_DIR) / cache_key

    if USE_PRECOMPUTED and cache_path.exists():
        print(f"[mds] load cached D_fit: {cache_path.name}")
        D_fit = np.load(cache_path)
    else:
        print(f"[mds] computing D_fit on {len(fit_idx)} points (of {len(big)}) …")
        D_fit = cosine_D_fit(Af)
        if USE_PRECOMPUTED:
            try:
                np.save(cache_path, D_fit)
                print(f"[mds] cached: {cache_path.name}")
            except Exception as e:
                print(f"[warn] cache save failed: {e}")

    mds  = MDS(**MDS_KW)
    Y_fit = mds.fit_transform(D_fit)                 # (m, 2)

    # OOS：分块 KNN-Shepard 插值（基于 1-cos 距离）
    Y = oos_place_by_knn_cos(A, fit_idx, Y_fit, k=OOS_K, p=OOS_P, block=4000)

    # 写回 + fitness 归一化（robust）
    big["mds_x"] = Y[:,0]; big["mds_y"] = Y[:,1]
    big["fitness_normed"] = robust_minmax(big["fitness_raw"].to_numpy(), q=(1,99))


    # ---- 分模型建面板缓存 ----
    for model, dfm in big.groupby("model"):
        key = (subtask, str(model))
        # 把该 subtask 的这个 model 面板累积到全局 all_panels
        all_panels.setdefault(key, pd.DataFrame())
        all_panels[key] = pd.concat(
            [all_panels[key], dfm[["generation","fitness_normed","mds_x","mds_y","model"]]],
            ignore_index=True
        )

    # # 画图（当前 subtask）
    # keys = sorted(panels.keys(), key=lambda x: x[1].lower())
    # n = len(keys)
    # ncols = min(5, n); nrows = math.ceil(n/ncols)
    # fig, axes = plt.subplots(nrows, ncols, figsize=(3.2*ncols, 2.8*nrows), squeeze=False)
    # fig.suptitle(f"Symbolic Regression — Task-level MDS ({subtask})\n(color=generation; size=fitness)", 
    #              fontsize=15, y=0.995)

    # # 颜色映射（该 subtask 的全局 generation）
    # g_all = pd.concat([panels[k]["generation"] for k in keys], ignore_index=True)
    # cmap = plt.get_cmap("viridis")
    # gnorm = Normalize(vmin=g_all.min(), vmax=g_all.max())

    # for i, key in enumerate(keys):
    #     r, c = divmod(i, ncols)
    #     ax = axes[r][c]
    #     dfp = panels[key].copy().sort_values("generation")

    #     cols  = cmap(gnorm(dfp["generation"].to_numpy(float)))
    #     sizes = 10 + 80 * dfp["fitness_normed"].to_numpy(float)

    #     ax.scatter(dfp["mds_x"], dfp["mds_y"], s=sizes, c=cols,
    #                edgecolor="white", linewidth=0.35, alpha=0.92, zorder=2)
    #     _, model = key
    #     ax.set_title(f"{subtask} | {model}", fontsize=10, pad=4)
    #     ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)

    # # 关空轴
    # for j in range(n, nrows*ncols):
    #     r, c = divmod(j, ncols)
    #     axes[r][c].set_axis_off()

    # # Generation colorbar
    # cax = fig.add_axes([0.92, 0.20, 0.015, 0.60])
    # cb  = mpl.colorbar.ColorbarBase(cax, cmap=cmap, norm=gnorm); cb.set_label("Generation", rotation=90)

    # # Size legend
    # legend_sizes = [0.2, 0.5, 0.8]
    # handles = [plt.scatter([], [], s=10+80*s, edgecolor="gray", facecolors="none", linewidth=0.8) for s in legend_sizes]
    # axes[0][0].legend(handles, [f"fitness≈{s:.1f}" for s in legend_sizes],
    #                   scatterpoints=1, frameon=True, fontsize=8, loc="upper left")

    # fig.subplots_adjust(left=0.06, right=0.90, top=0.90, bottom=0.06, wspace=0.12, hspace=0.12)
    # plt.show()
# ========== 画图：所有 (subtask, model) ==========
keys = sorted(all_panels.keys(), key=lambda x: (x[0], x[1].lower()))
if not keys:
    print("[warn] nothing to plot.")
else:
    n = len(keys)
    ncols = min(5, n); nrows = math.ceil(n/ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.2*ncols, 2.8*nrows), squeeze=False)
    fig.suptitle("Symbolic Regression (Oscillator1 / Oscillator2) — Novelty at the Semantic Level", fontsize=16, y=0.995)

    # 颜色映射（全局 generation）
    g_all = pd.concat([all_panels[k]["generation"] for k in keys], ignore_index=True)
    cmap = plt.get_cmap("viridis")
    gnorm = Normalize(vmin=g_all.min(), vmax=g_all.max())

    for i, key in enumerate(keys):
        r, c = divmod(i, ncols)
        ax = axes[r][c]
        dfp = all_panels[key].copy().sort_values("generation")

        cols  = cmap(gnorm(dfp["generation"].to_numpy(float)))
        sizes = 10 + 80 * dfp["fitness_normed"].to_numpy(float)

        # 底层密度（可选）
        # ax.hexbin(dfp["mds_x"], dfp["mds_y"], gridsize=40, mincnt=3, alpha=0.22)

        ax.scatter(dfp["mds_x"], dfp["mds_y"], s=sizes, c=cols,
                   edgecolor="white", linewidth=0.35, alpha=0.92, zorder=2)
        subtask, model = key
        ax.set_title(f"{subtask} | {model}", fontsize=10, pad=4)
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)

    # 关空轴
    for j in range(n, nrows*ncols):
        r, c = divmod(j, ncols)
        axes[r][c].set_axis_off()

    # Generation colorbar
    cax = fig.add_axes([0.92, 0.20, 0.015, 0.60])
    cb  = mpl.colorbar.ColorbarBase(cax, cmap=cmap, norm=gnorm); cb.set_label("Generation", rotation=90)

    # 尺寸图例 (Fitness)
    legend_sizes = [0.2, 0.5, 0.8]
    handles = [plt.scatter([], [], s=10 + 80 * s, edgecolor="gray", facecolors="none", linewidth=0.8)
               for s in legend_sizes]
    labels = [f"fitness≈{s:.1f}" for s in legend_sizes] # 先定义 labels
    
    # --- ✅ 修改：将图例放在整个大图的右上角区域 ---
    # (我们将其放置在 Colorbar 的正上方)
    fig.legend(handles, labels,
               scatterpoints=1, 
               frameon=True, 
               fontsize=10, 
               loc="upper left",  # 将图例的“左上角”...
               bbox_to_anchor=(0.90, 0.93), # ...锚定到 (x=92%, y=90%) 的位置
               # title="Fitness", # 可以给图例加个标题
              )

    fig.subplots_adjust(left=0.06, right=0.90, top=0.93, bottom=0.06, wspace=0.12, hspace=0.12)
    plt.savefig("/Users/lievretre/Desktop/evo_eval_plots_10_28/novelty_mds_sr.png",dpi=300)
    plt.show()